# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. The dataset schema follows the Croissant specification, enabling programmatic access to structured metadata and data tables.

### Dataset Source
This dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

Let's load the metadata and list the available record sets from the FAIR^2 dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print("\033[1mDataset Name:\033[0m", metadata.name)
print("\033[1mDescription:\033[0m", metadata.description)
print("\033[1mPublished:\033[0m", getattr(metadata, 'datePublished', 'N/A'))
print("\033[1mAuthors' @id:\033[0m", [author['@id'] if isinstance(author, dict) and '@id' in author else author for author in getattr(metadata, 'author', [])])

## 2. Data Overview

Explore the available record sets, their `@id`s, and their fields. All dataset entities are referenced by their `@id` for precise access and manipulation.

> **Tip:** Use the dataset's record_sets attribute to discover all available record sets and their field structure.

In [ ]:
# List available record sets and their fields by @id

record_sets = ds.record_sets
if not record_sets:
    print("No record sets found. Please check the dataset schema or contact the dataset maintainer.")
else:
    print("\033[1mRecord set overview:\033[0m")
    for rs in record_sets:
        print(f"  - RecordSet name: {rs.name}")
        print(f"    @id: {rs.id}")
        # List fields and their @ids
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for field in rs.fields:
                print(f"      - {field.name} (\"@id\": {field.id})  [type: {field.data_type}]")
        print()

## 3. Data Extraction

Load data from each record set using its `@id`. Fields (columns) will also be referenced by their `@id`. The loaded data is saved in a dictionary of DataFrames for further analysis.

In [ ]:
# Automatically collect record set @ids for extraction
# If no record sets exist, skip further extraction

dataframes = {}
record_set_ids = [rs.id for rs in ds.record_sets] if ds.record_sets else []

if not record_set_ids:
    print("[INFO] No record sets available in the dataset.")
else:
    print("\033[1mExtracting data from record sets...\033[0m\n")
    for rs_id in record_set_ids:
        try:
            records = list(ds.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id: {rs_id}")
        except Exception as e:
            print(f"[ERROR] Could not load records for RecordSet {rs_id}: {e}")
    if dataframes:
        # Select first non-empty dataframe for demo
        for rs_id, df in dataframes.items():
            if not df.empty:
                print(f"\nSample columns from RecordSet {rs_id}: \n", df.columns.tolist())
                display(df.head())
                demo_rs_id = rs_id
                break

## 4. Exploratory Data Analysis (EDA)

Let's perform some EDA: filter records based on a numeric field, normalize a variable, and group data.

For this section, you'll need to:
1. Choose a RecordSet of interest (by `@id`).
2. Pick a numeric field (by `@id`) within that RecordSet.
3. Optionally, select a grouping field for aggregation.

> Below, the code dynamically selects a numeric field from your chosen record set for demonstration.

In [ ]:
import numpy as np

if dataframes:
    # Use the demo_rs_id set above as default
    df = dataframes[demo_rs_id]
    print(f"\033[1mWorking with RecordSet @id:\033[0m {demo_rs_id}")

    # Try to automatically find first numeric column
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric field found in this RecordSet.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} found")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to select a group field (categorical), excluding the numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == object or df[col].dtype.name == "category"):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} by {group_field} (grouped):")
            print(grouped_df.head())
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Let's visualize data distributions and relationships between key fields. We'll use matplotlib and seaborn for plotting. Remember, use fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    # Histogram of the normalized numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=20)
    plt.title(f'Normalized distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id + ' (normalized)')
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, boxplot of the numeric field by group
    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we:
* Loaded the FAIR^2 dataset metadata and extracted key information such as dataset name, description, and authors.
* Explored available record sets and demonstrated field/column referencing by `@id`.
* Extracted data from one or more record sets, and performed exploratory data analysis including filtering, normalization, and grouping based on field types.
* Visualized the distribution of a numeric variable and explored its relationship to a grouping field.

Use this notebook as a starting point for deeper exploration, statistical modeling, or policy analysis using the FAIR^2 rangeland adoption dataset.